In [ ]:
import os
import time
from typing import Optional
import nemo_run as run

# TRAINING_SCRIPT = "preprocess_c4_save.py"
TRAINING_SCRIPT = "train_def_1024_c4.py"


def slurm_executor(
    user: str,
    host: str,
    identity_file: str,
    remote_job_dir: str,
    account: str,
    partition: str,
    nodes: int,
    devices: int,
    time: str = "11:00:00",
    custom_env_vars: Optional[dict[str, str]] = None,
    retries: int = 0,
) -> run.SlurmExecutor:
    if not (user and host and remote_job_dir and account and partition and nodes and devices):
        raise RuntimeError(
            "Please set user, host, remote_job_dir, account, partition, nodes, and devices args for using this function."
        )

    # Env vars for jobs are configured here
    env_vars = {
        "TORCH_NCCL_AVOID_RECORD_STREAMS": "1",
        "NCCL_NVLS_ENABLE": "0",
        "NVTE_DP_AMAX_REDUCE_INTERVAL": "0",
        "NVTE_ASYNC_AMAX_REDUCTION": "1",
    }
    if custom_env_vars:
        env_vars |= custom_env_vars

    # This will package the train.py script in the current working directory to the remote cluster.
    # If you are inside a git repo, you can also use https://github.com/NVIDIA/NeMo-Run/blob/main/src/nemo_run/core/packaging/git.py.
    # If the script already exists on your container and you call it with the absolute path, you can also just use `run.Packager()`.
    packager = run.PatternPackager(include_pattern=[TRAINING_SCRIPT, "./data/**"], relative_path=[os.getcwd(), os.getcwd()]) # , "./slimpajama/**"

    # This defines the slurm executor.
    # We connect to the executor via the tunnel defined by user, host and remote_job_dir.
    executor = run.SlurmExecutor(
        account=account,
        partition=partition,
        tunnel=run.SSHTunnel(
            user=user,
            host=host,
            job_dir=remote_job_dir, # This is where the results of the run will be stored by default.
            identity=identity_file # OPTIONAL: Provide path to the private key that can be used to establish the SSH connection without entering your password.
        ),
        nodes=nodes,
        ntasks_per_node=devices,
        gpus_per_node=devices,
        exclusive=True,
        # gres="gpu:4",
        packager=packager,
    )

    executor.env_vars = env_vars
    executor.retries = retries
    executor.time = time
    return executor

# Run it locally
# executor = run.LocalExecutor()
executor = slurm_executor(
    user="plgmstefaniak",
    host="helios.cyfronet.pl",
    identity_file="/home/maciej/.ssh/plgrid",
    remote_job_dir="/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary",
    # remote_job_dir="net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/llm_random_cemetery",
    # remote_job_dir="/net/home/plgrid/plgmstefaniak/tmp/",
    account="plgllmefficont2-gpu-gh200",
    partition="plgrid-gpu-gh200",
    nodes=1,
    devices=4,

) # pass in args relevant to your cluster

title = "nemo_2_training_experiment"
exp_id = f"{title}_{int(time.time())}"
with run.Experiment(title, id=exp_id, log_level="INFO") as exp: # bug when you specigy id, expected id = {title}_{id}
    training_job = run.Script(
    inline=f"""
        python {TRAINING_SCRIPT}
        """,
        entrypoint = f"""singularity exec --nv \
        --bind /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/{title}/{exp_id}/training:/nemo_run \
        --bind /net/scratch/hscra/plgrid/plgmaciejpioro/c4/train:/nemo_run/datasets/c4/train \
        --bind /net:/net \
        --pwd /nemo_run/code \
        /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/images/nemorand_dev.sif \
        bash"""
        # entrypoint = """
        # pwd ."""
    )
    exp.add(training_job, executor=executor, tail_logs=True, name="training")
    # Add more jobs as needed

    # --bind /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744659529/training/code/preprocessing_results:/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744659529/training/code/preprocessing_results \
    # --bind /net:/net \
    # Run the experiment
    exp.run(detach=False)

# /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744651579/training/code/c4_en_train.jsonl
# /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744659529/training

────────── Entering Experiment nemo_2_training_experiment with id: nemo_2_training_experiment_1744670273 ──────────

[00:37:53] Connecting to plgmstefaniak@helios.cyfronet.pl                                             ]8;id=639228;file:///home/maciej/projects/cluster/nemorand/nemo_run/core/tunnel/client.py\client.py]8;;\:]8;id=552580;file:///home/maciej/projects/cluster/nemorand/nemo_run/core/tunnel/client.py#257\257]8;;\

Connected (version 2.0, client OpenSSH_8.0)
Authentication (publickey) successful!
rsyncing /home/maciej/.nemo_run/experiments/nemo_2_training_experiment/nemo_2_training_experiment_1744670273 to /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment ...
Successfully ran `rsync  -pthrvz  --rsh='ssh -i /home/maciej/.ssh/plgrid -p 22 ' /home/maciej/.nemo_run/experiments/nemo_2_training_experiment/nemo_2_training_experiment_1744670273 plgmstefaniak@helios.cyfronet.pl:/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment`


[00:37:57] Launching job training for experiment nemo_2_training_experiment                       ]8;id=555187;file:///home/maciej/projects/cluster/nemorand/nemo_run/run/experiment.py\experiment.py]8;;\:]8;id=19854;file:///home/maciej/projects/cluster/nemorand/nemo_run/run/experiment.py#744\744]8;;\

Launched app: slurm_tunnel://nemo_run/402164


...

Task 0: training
- Status: PENDING
- Executor: SlurmExecutor on plgmstefaniak@helios.cyfronet.pl
- Job id: 402164
- Local Directory: /home/maciej/.nemo_run/experiments/nemo_2_training_experiment/nemo_2_training_experiment_1744670273/training
- Remote Directory: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744670273/training

Waiting for job 402164 to finish [log=True]...


[00:38:04] Waiting for app state response before fetching logs...                                       ]8;id=344791;file:///home/maciej/projects/cluster/nemorand/nemo_run/run/logs.py\logs.py]8;;\:]8;id=2256;file:///home/maciej/projects/cluster/nemorand/nemo_run/run/logs.py#105\105]8;;\

[chan 13] Opened sftp connection (server version 3)


training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 15:4: not a valid test operator:  
training/0 15:4: not a valid test operator: 12.8
training/0 21:4: not a valid test operator: (
training/0 21:4: not a valid test operator: 565.57.01
training/0 15:4: not a valid test operator:  
training/0 15:4: not a valid test operator: 12.8
training/0 21:

In [ ]:
# srun 
# --output /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2.0_training_experiment/nemo_2.0_training_experiment_1744148528/training/log-plgllmefficont2-gpu-gh200-plgllmefficont2-gpu-gh200.training_%j_${SLURM_RESTART_COUNT:-0}.out 
# --container-mounts /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2.0_training_experiment/nemo_2.0_training_experiment_1744148528/training:/nemo_run 
# --container-workdir /nemo_run/code 
# --wait=60 --kill-on-bad-exit=1 bash /nemo_run/scripts/training.sh

In [ ]:
import nemo_run as run

experiment = run.Experiment.from_id(exp_id)                                 
experiment.status() # Gets the overall status                                                                      
experiment.logs("training") # Gets the log for the provided task                                                   
# experiment.cancel("training") # Cancels the provided task if still running

# /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/1744234741/training
# /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2.0_training_experiment/1744234741/training

[00:25:45] Connecting to plgmstefaniak@helios.cyfronet.pl                                             ]8;id=648647;file:///home/maciej/projects/cluster/nemorand/nemo_run/core/tunnel/client.py\client.py]8;;\:]8;id=926693;file:///home/maciej/projects/cluster/nemorand/nemo_run/core/tunnel/client.py#257\257]8;;\

Connected (version 2.0, client OpenSSH_8.0)
Authentication (publickey) successful!


Experiment Status for nemo_2_training_experiment_1744669354

Task 0: training
- Status: FAILED
- Executor: SlurmExecutor on plgmstefaniak@helios.cyfronet.pl
- Job id: 402154
- Local Directory: /home/maciej/.nemo_run/experiments/nemo_2_training_experiment/nemo_2_training_experiment_1744669354/training
- Remote Directory: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744669354/training

[00:25:48] Fetching logs for training                                                             ]8;id=949738;file:///home/maciej/projects/cluster/nemorand/nemo_run/run/experiment.py\experiment.py]8;;\:]8;id=753186;file:///home/maciej/projects/cluster/nemorand/nemo_run/run/experiment.py#931\931]8;;\

[chan 7] Opened sftp connection (server version 3)
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 INFO:    underlay of /usr/bin/nvidia-smi required more than 50 (508) bind mounts
training/0 FATAL:   container creation failed: mount hook function failure: hook function for tag prelayer returns error: failed to create /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experi

In [ ]:
experiment

Graphviz rendering failed: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH
